In [1]:
import pandas as pd
import igraph
import pickle as pk
import os

In [2]:
print("Available network graph files:")
for g in set( [g.split('_')[0] for g in os.listdir("graphs/")] ):
    print("-", g)

Available network graph files:
- Humanities
- LitLangMed


In [3]:
dataset = 'LitLangMed'

# Create cocitation graph

In [4]:
def describe(g):
    print( g.vcount() )
    print( g.ecount() )
    print( g.vertex_attributes() )
    print( g.edge_attributes() )

In [5]:
## Load graph
g = igraph.Graph.Read_Pickle( 'graphs/' + dataset + '_graph.pk' )
describe(g)

5297
6428
['name', 'title', 'school', 'type', 'authors']
[]


In [6]:
cocitation_matrix = g.cocitation()

In [7]:
gc = igraph.Graph.Weighted_Adjacency(
    cocitation_matrix,
    mode='undirected',
    attr='weight',
    loops=False
)

for attr in g.vertex_attributes():
    gc.vs[attr] = g.vs[attr]

In [8]:
describe(gc)

5297
886977
['name', 'title', 'school', 'type', 'authors']
['weight']


In [9]:
## Write to pickle
gc.write_pickle( "graphs/" + dataset + "_cocitation_graph.pk")

# Visualize

In [10]:
from fa2_modified import ForceAtlas2
import random

In [11]:
gc = igraph.Graph.Read_Pickle( 'graphs/' + dataset + '_cocitation_graph.pk' )

### Filter and layout

In [18]:
filt_thresh = 5

In [19]:
# Delete edges with cocitation weight less than threshold
gc.es.select(weight_lt = filt_thresh).delete()

# Delete vertices with zero degree
gc.vs.select(_degree_lt=1).delete()

print( gc.vcount() )
print( gc.ecount() )

3628
15773


In [20]:
random.seed(1327)

forceatlas2 = ForceAtlas2(
                        # Behavior alternatives
                        outboundAttractionDistribution=False,  # Dissuade hubs
                        linLogMode=False,  # NOT IMPLEMENTED
                        adjustSizes=False,  # Prevent overlap (NOT IMPLEMENTED)
                        edgeWeightInfluence=1.5,

                        # Performance
                        jitterTolerance=1.0,  # Tolerance
                        barnesHutOptimize=True,
                        barnesHutTheta=1.2,
                        multiThreaded=False,  # NOT IMPLEMENTED

                        # Tuning
                        scalingRatio=1.0,
                        strongGravityMode=True,
                        gravity=2,

                        # Log
                        verbose=True)

l = forceatlas2.forceatlas2_igraph_layout( gc, pos=None, iterations=1000, weight_attr='weight' )

100%|██████████████████████████████████████████████████████████| 1000/1000 [00:35<00:00, 28.11it/s]

BarnesHut Approximation  took  16.84  seconds
Repulsion forces  took  17.36  seconds
Gravitational forces  took  0.01  seconds
Attraction forces  took  0.48  seconds
AdjustSpeedAndApplyForces step  took  0.42  seconds


In [21]:
# Write layout
with open("layouts/" + dataset + "_" + str(filt_thresh) +  ".pkl", "wb") as f:
    pk.dump(l, f)

### Plot

In [22]:
with open("layouts/" + dataset + "_" + str(filt_thresh) + ".pkl", "rb") as f:
    l = pk.load(f)

In [23]:
igraph.plot(
    gc,
    'plots/' + dataset + "_" + str(filt_thresh) + '_cocitations.png', 
    layout=l,
    vertex_size = 1,
    #vertex_size = [(s/200)+2 for s in gc.vs['strength']],
    edge_width=0.1
)